# 🔥 **CellPatcher for PyTorch**
___

This notebook loads the patches of a **CellPatcher export** as **PyTorch tensors**, the way a
training or feature-extraction script reads them. It reads the output folder the **CellPatcher
notebook** writes in its step 4 (**patches.csv**, **images.csv**, **run_config.json** and the patch
files) with CellPatcher's `ExportedPatchTorchDataset`, and shows the first batch a `DataLoader`
gives: its shape, type, value range and images. Patches exported with raw values are normalized on
load with the parameters stored in **patches.csv**, exactly as the CellPatcher notebook would have
normalized them.

### Quick workflow

1. Run the **Step 0** cell to start.
2. **Step 1:** paste the CellPatcher output folder and press **Load the patches**.

### Before you start

- [ ] You exported patches with the **CellPatcher notebook** (step 4). To stack them into batches
  they need one size: a fixed patch size or **Resize patches to** in its step 2.

> 🛡️ **Nothing is written:** the patches are only read.

___
## 🧰 **Step 0: Start the notebook**

Click the cell below and press **Shift + Enter** (or the ▶ button in the toolbar). A green box
appears when everything is ready.

> **Colab users:** Step 0 connects your Google Drive (allow access in the window that opens)
> and installs CellPatcher **v0.2.0rc1** from GitHub, which takes a minute or two; Colab already
> has PyTorch. Keep the CellPatcher output folder on Google Drive: in this notebook your Drive is
> the folder **/content/drive/MyDrive**. Everything else on the runtime is wiped when it
> disconnects; then run Step 0 again.
>
> **JupyterLab users:** Step 0 uses CellPatcher as installed for this notebook's kernel (or the
> copy of the repository around the notebook), installs what is missing (PyTorch included, a large
> download if it is not installed yet), and prints the Python version and working folder. Nothing
> is mounted.

> 💡 **Tip:** the code of each cell is hidden; you only need to run the cells, from top to bottom,
> one after the other.

In [ ]:
# @title Run to start the CellPatcher PyTorch notebook
import platform
import subprocess
import sys
from importlib import import_module, invalidate_caches
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec
from pathlib import Path

# The notebook's version and name, as LabConstrictor's version check reads them
# (notebooks/notebook_latest_versions.yaml lists the latest version).
current_version = "0.2.0rc1"
notebook_name = "CellPatcher_Pytorch"
# The release installed from GitHub when the package is missing (Google Colab, or a
# downloaded notebook). Set to the new tag, with current_version, before each release is
# tagged, so a tagged notebook installs its own release.
RELEASE_TAG = "v0.2.0rc1"
REPOSITORY = "https://github.com/CellMigrationLab/CellPatcher.git"
# The package's extras to install; on Colab also zarr, to read OME-Zarr images.
EXTRAS = "notebook,torch"
# The notebook's interface in the package, as "module:class".
INTERFACE = "cellpatcher.notebook_torch:TorchNotebook"

if "google.colab" in sys.modules:
    # Colab only: Google Drive holds the data and the results, which outlive the runtime.
    from google.colab import drive, output

    print(
        "Google Colab: connecting Google Drive. Allow access in the window that opens; "
        "your Drive is then the folder /content/drive/MyDrive.",
        flush=True,
    )
    try:
        drive.mount("/content/drive")
    except Exception as exc:  # a refused or interrupted authorisation
        raise RuntimeError(
            "Google Drive could not be connected. Run Step 0 again and allow access to "
            "your Drive; your data and output folders must be on Drive."
        ) from exc
    # Colab ships ipywidgets 7; this notebook's widgets need ipywidgets 8's front end.
    output.enable_custom_widget_manager()
    EXTRAS += ",zarr"
else:
    print(
        f"Local Jupyter: Python {platform.python_version()} on {platform.platform()}\n"
        f"Working folder: {Path.cwd()}",
        flush=True,
    )


def _missing_requirements() -> list[str]:
    """Return the notebook's requirements this kernel cannot import (or too old)."""
    required = (
        "cellpatcher",
        "morphobricks_preprocessing",
        "morphobricks_normalization",
        "numpy",
        "tifffile",
        "ipywidgets",
        "matplotlib",
        *(extra for extra in ("zarr", "torch") if extra in EXTRAS.split(",")),
    )
    missing = [name for name in required if find_spec(name) is None]
    try:
        widgets_major = int(version("ipywidgets").split(".")[0])
    except (PackageNotFoundError, ValueError):
        widgets_major = 8
    if widgets_major < 8 and "ipywidgets" not in missing:
        missing.append("ipywidgets 8")
    return missing


def _install_target() -> tuple[list[str], str]:
    """Return what pip installs when something is missing, and where CellPatcher comes from."""
    current = Path.cwd().resolve()
    checkout = next(
        (
            folder
            for folder in (current, *current.parents)
            if (folder / "setup.py").is_file() and (folder / "src/cellpatcher/__init__.py").is_file()
        ),
        None,
    )
    if checkout is not None:
        source = checkout / "src"
        loaded = sys.modules.get("cellpatcher")
        if loaded is not None and not Path(loaded.__file__).resolve().is_relative_to(source):
            raise RuntimeError(
                "This kernel has already loaded another copy of CellPatcher. "
                "Restart the kernel, then run Step 0 to use this checkout."
            )
        # A local source tree works immediately, including in a git worktree.
        sys.path[:] = [str(source), *(entry for entry in sys.path if entry != str(source))]
        invalidate_caches()
        return ["-e", f"{checkout}[{EXTRAS}]"], f"the local checkout at {checkout}"
    if find_spec("cellpatcher") is not None:
        try:
            return [f"cellpatcher[{EXTRAS}]"], f"the installed package, version {version('cellpatcher')}"
        except PackageNotFoundError:
            return [f"cellpatcher[{EXTRAS}]"], "the installed package"
    return [f"cellpatcher[{EXTRAS}] @ git+{REPOSITORY}@{RELEASE_TAG}"], f"release {RELEASE_TAG} from GitHub"


def _start_notebook() -> object:
    """Prepare this kernel's environment and return the notebook interface."""
    # This standalone cell can run before the package's Python requirement is checked.
    if sys.version_info < (3, 11):  # noqa: UP036
        raise RuntimeError("Select a Python 3.11 or newer notebook kernel, then run Step 0 again.")

    environment = "Google Colab" if "google.colab" in sys.modules else "local Jupyter"
    print(f"Checking the {environment} environment...", flush=True)
    install_target, origin = _install_target()
    missing = _missing_requirements()
    if missing:
        print(f"Installing notebook requirements ({', '.join(missing)})...", flush=True)
        try:
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", *install_target],
                check=True,
            )
        except (subprocess.CalledProcessError, OSError) as exc:
            raise RuntimeError(
                "Notebook setup could not install its requirements. Check the installation "
                "output above, internet access and write permissions, then rerun Step 0. "
                "Installing from GitHub also needs Git."
            ) from exc
        invalidate_caches()
        missing = _missing_requirements()
        if missing:
            raise RuntimeError(
                f"Installation finished, but this kernel still cannot find {', '.join(missing)}. "
                "Restart the kernel and run Step 0 again."
            )
    loaded_widgets = str(getattr(sys.modules.get("ipywidgets"), "__version__", "8")).split(".")[0]
    if loaded_widgets.isdigit() and int(loaded_widgets) < 8:
        raise RuntimeError(
            "This kernel loaded ipywidgets 7 before ipywidgets 8 was installed. Restart the "
            "kernel (on Colab: Runtime > Restart session), then run Step 0 again."
        )

    print(f"Using {origin}.\nNotebook {notebook_name} {current_version}; Python: {sys.executable}", flush=True)
    module, _, name = INTERFACE.partition(":")
    try:
        interface = import_module(module)
    except ImportError as exc:
        raise RuntimeError(
            "CellPatcher was found, but the notebook interface could not be loaded. "
            "Read the original error below. If packages were changed in this session, "
            "restart the kernel and run Step 0 again."
        ) from exc
    return getattr(interface, name)()


notebook = _start_notebook()
notebook.welcome()

___
## 📦 **Step 1: Load the patches as tensors**

Run the cell below, then:

1. Paste the path of the **CellPatcher output folder** (the one holding **patches.csv**), or click
   **Browse…**.
2. Choose how many **Patches in the batch**.
3. Press **Load the patches**. A green card gives the number of patches and the batch tensor:
   `(patches, channels, height, width)` with every channel and value as exported (normalized if the
   export was), its type and its range; the first channel of each patch is drawn below. A red card says what to do, for example export the patches with one size.

> **Colab users:** paste a path that starts with **/content/drive/MyDrive/**. **Browse…** cannot
> open a dialog on Colab: open the **Files** panel in the left bar, right-click the folder, choose
> **Copy path** and paste it.
>
> **JupyterLab users:** paste the path or use **Browse…**, which opens your computer's folder
> dialog.

In [ ]:
# @title Run to load the patches
notebook.patches_step()

___
## ✅ **After running: use the patches in your own script**

The same dataset in a training or feature-extraction script (the folder is the one of step 1):

```python
from torch.utils.data import DataLoader
from cellpatcher.torch_dataset import ExportedPatchTorchDataset, PatchTensorConfig

dataset = ExportedPatchTorchDataset(
    "my_experiment_CellPatcher/patches.csv",
    axes="CYX",  # the output_axes column of patches.csv (step 1 shows it)
    # every channel and value as exported (no channel repeated, no integer scaling)
    tensor_config=PatchTensorConfig(channel_policy="keep", scale_policy="none"),
)
loader = DataLoader(dataset, batch_size=8, shuffle=True)
for batch in loader:
    images = batch["image"]        # (patches, channels, height, width), float32
    patch_ids = batch["patch_id"]  # join your results to patches.csv on patch_id
```

Each item also holds its **patches.csv** row (`batch["patch"]`): condition, biological repeat, image,
coordinates, ... Keep the export folder together: the table, its **run_config.json** and the patch
files.

More help: the [documentation](https://github.com/CellMigrationLab/CellPatcher/blob/main/docs/README.md).